In [2]:
# Import the libraries
import numpy as np
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter

In [3]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/23ae2440-d9b7-4bf1-b490-
[nltk_data]     439ad9e57e22/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/23ae2440-d9b7-4bf1-b490-
[nltk_data]     439ad9e57e22/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/23ae2440-d9b7-4bf1-b490-
[nltk_data]     439ad9e57e22/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"] 

In [5]:
# Text Preprocessing
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # 3a: Lowercase
    text = text.lower()
    
    # 3b: Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 3c: Tokenize
    tokens = word_tokenize(text)
    
    # 3d: Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    
    # 3e: Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    
    return ' '.join(tokens)

preprocessed_dataset = [preprocess(doc) for doc in dataset]

In [6]:
vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(preprocessed_dataset)

In [7]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Display the document and its predicted cluster in a table 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(preprocessed_dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))
# Print top terms per cluster 
print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 
for i in range(k): 
 print("Cluster %d:" % i) 
 for ind in order_centroids[i, :10]: 
  print(' %s' % terms[ind]) 
 print() 

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Document                        Predicted Cluster
----------------------------  -------------------
love play footbal weekend                       1
enjoy hike camp mountain                        0
like read book watch movi                       1
prefer play video game sport                    1
love listen music go concert                    1

Top terms per cluster:
Cluster 0:
 mountain
 hike
 enjoy
 camp
 weekend
 read
 sport
 video
 watch
 movi

Cluster 1:
 play
 love
 footbal
 weekend
 video
 concert
 game
 sport
 music
 listen



In [8]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Purity: 0.8
